# CAMUS inference visualisation

Loads a trained checkpoint from `runs/` and looks at what it actually predicts:
side-by-side overlays, boundary contours, error maps, per-case metrics from
`SegmentationEvaluator`, and the cases the model gets worst.

Numbers alone hide the failure modes that matter clinically — a Dice of 0.93 on
the LV cavity says nothing about whether the endocardial border is systematically
inside or outside the true one, and that bias is what propagates into volume and
ejection fraction. Everything here is meant to be looked at, not averaged.

Labels: `1` LV endocardium (cavity) · `2` LV myocardium · `3` left atrium.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D
from monai.metrics import compute_dice
from monai.networks.utils import one_hot
from torch.utils.data import DataLoader

from camus_dataset import LABELS, NUM_CLASSES, CAMUSDataset
from evaluation import METRIC_NAMES, SegmentationEvaluator
from model import build_unet, count_parameters, resolve_device

DATA_ROOT = Path.cwd().parent / "CAMUS_public"
RUN_DIR = Path("runs/unet_baseline")
CHECKPOINT = RUN_DIR / "best.pt"      # "last.pt" for the final epoch instead
SPLIT = "test"                        # the official held-out patients

assert DATA_ROOT.is_dir(), f"CAMUS_public not found at {DATA_ROOT} - fix DATA_ROOT"
assert CHECKPOINT.is_file(), f"no checkpoint at {CHECKPOINT} - train.py writes it"

plt.rcParams["figure.dpi"] = 110
DEVICE = resolve_device()
print(f"data root   {DATA_ROOT}")
print(f"checkpoint  {CHECKPOINT}")
print(f"device      {DEVICE}")

## 1. Load the checkpoint

`train.py` saves the config alongside the weights, so the model and the data
pipeline can be rebuilt exactly as they were trained rather than from defaults
that may have drifted since.

In [ ]:
checkpoint = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=True)
cfg = checkpoint["config"]

model = build_unet().to(DEVICE)
model.load_state_dict(checkpoint["model"])
model.eval()

print(f"epoch       {checkpoint['epoch']}")
print(f"val_dice    {checkpoint['val_dice']:.4f}   (selection metric, validation split)")
print(f"parameters  {count_parameters(model):,}")
print(f"image_size  {cfg['image_size']}   batch_size {cfg['batch_size']}   lr {cfg['lr']}")

## 2. Test split

The image size has to match training: the network is fully convolutional and
would run at other sizes, but the receptive field was tuned at `cfg["image_size"]`
and the spacing correction in the dataset keeps millimetre metrics honest only if
the resize is the same one the model was trained under.

In [ ]:
image_size = (cfg["image_size"], cfg["image_size"])
test_ds = CAMUSDataset(DATA_ROOT, split=SPLIT, image_size=image_size)
test_loader = DataLoader(test_ds, batch_size=cfg["batch_size"], shuffle=False, num_workers=0)

print(test_ds)


def find_index(dataset, key):
    """Position of a `patientXXXX_VIEW_INSTANT` key in `dataset`."""
    for i, sample_key in enumerate(dataset.samples):
        if str(sample_key) == key:
            return i
    raise KeyError(f"{key} is not in the {dataset.split} split")


def sample_by_key(dataset, key):
    return dataset[find_index(dataset, key)]

## 3. Inference helpers

`predict` returns probabilities as well as the hard label map: the softmax
confidence is what section 8 uses to show *where* the model is unsure, which is
usually the same place the boundary is wrong.

In [ ]:
@torch.inference_mode()
def predict(sample):
    """Run the model on one dataset sample.

    Returns `(pred, probs)` with `pred` a LongTensor (H, W) and `probs` a
    FloatTensor (C, H, W) of softmax probabilities, both on the CPU.
    """
    logits = model(sample["image"][None].to(DEVICE))
    probs = logits.softmax(dim=1)[0].cpu()
    return probs.argmax(0), probs


def per_class_dice(pred, label):
    """Dice per foreground class for a single case, as {name: value}.

    NaN means the class is absent from the ground truth (`ignore_empty`), which
    is a fact about the case rather than a failure of the model.
    """
    pred_oh = one_hot(pred[None, None].long(), num_classes=NUM_CLASSES)
    target_oh = one_hot(label[None, None].long(), num_classes=NUM_CLASSES)
    dice = compute_dice(pred_oh, target_oh, include_background=False, ignore_empty=True)[0]
    return {LABELS[c]: float(dice[c - 1]) for c in range(1, NUM_CLASSES)}


def dice_caption(scores):
    return "  ".join(f"{name.split('_')[-1]} {value:.3f}" for name, value in scores.items())

## 4. Plotting helpers

Five panels per case, each answering a different question:

| panel | question |
| --- | --- |
| B-mode | is the image even readable? |
| ground truth | what did the expert draw? |
| prediction | what did the model draw? |
| contours | where do the two borders diverge, and in which direction? |
| errors | how much area is over- vs under-segmented? |

The contour panel is the one to trust for boundary bias — filled overlays make a
2 px systematic offset invisible, and 2 px is ~0.6 mm of endocardium.

In [ ]:
CLASS_COLORS = {1: "#e8564a", 2: "#2fb8a0", 3: "#4a7fe8"}
ERROR_COLORS = {"false positive": "#e8564a", "false negative": "#4a7fe8"}


def overlay_labels(ax, label, alpha=0.35, contours=True, linestyle="-"):
    """Paint each class over the image, plus a crisp contour on the boundary.

    Classes are drawn outermost first so the endocardial border ends up on top:
    it is shared with the inner edge of the myocardium, and whichever is drawn
    last is the only one visible - which must be the border LV volume and EF
    are measured from.
    """
    for cls, color in reversed(CLASS_COLORS.items()):
        mask = label == cls
        if not mask.any():
            continue
        if alpha:
            rgba = np.zeros((*mask.shape, 4))
            rgba[mask] = (*to_rgb(color), alpha)
            ax.imshow(rgba, interpolation="nearest")
        if contours:
            ax.contour(mask.astype(float), levels=[0.5], colors=[color],
                       linewidths=1.1, linestyles=linestyle)


def draw_error_map(ax, pred, label):
    """Over-segmentation vs under-segmentation, pooled over foreground classes.

    Red is predicted foreground the expert left out, blue is expert foreground
    the model missed. Class confusion inside the foreground is deliberately not
    shown here - the contour panel covers that - so the two colours read as a
    single question: is the model too generous or too conservative?
    """
    pred_fg = (pred > 0).numpy()
    true_fg = (label > 0).numpy()
    rgba = np.zeros((*pred_fg.shape, 4))
    rgba[pred_fg & ~true_fg] = (*to_rgb(ERROR_COLORS["false positive"]), 0.75)
    rgba[~pred_fg & true_fg] = (*to_rgb(ERROR_COLORS["false negative"]), 0.75)
    ax.imshow(rgba, interpolation="nearest")
    return float((pred_fg & ~true_fg).sum()), float((~pred_fg & true_fg).sum())


def class_legend(dashed_pred=True):
    handles = [Line2D([0], [0], color=CLASS_COLORS[c], lw=3, label=f"{c} {LABELS[c]}")
               for c in CLASS_COLORS]
    if dashed_pred:
        handles += [
            Line2D([0], [0], color="0.35", lw=1.5, ls="-", label="ground truth"),
            Line2D([0], [0], color="0.35", lw=1.5, ls="--", label="prediction"),
        ]
    return handles


def error_legend():
    return [Line2D([0], [0], color=color, lw=3, label=name)
            for name, color in ERROR_COLORS.items()]


def show_prediction(sample, pred=None, probs=None):
    """The five-panel view for one case."""
    if pred is None:
        pred, probs = predict(sample)
    image = sample["image"][0].numpy()
    label = sample["label"]
    scores = per_class_dice(pred, label)

    fig, axes = plt.subplots(1, 5, figsize=(16, 3.9), layout="constrained")
    for ax in axes:
        ax.imshow(image, cmap="gray")
        ax.set_xticks([]); ax.set_yticks([])

    axes[0].set_title("B-mode", fontsize=9)

    overlay_labels(axes[1], label.numpy())
    axes[1].set_title("ground truth", fontsize=9)

    overlay_labels(axes[2], pred.numpy())
    axes[2].set_title("prediction", fontsize=9)

    overlay_labels(axes[3], label.numpy(), alpha=0.0, linestyle="-")
    overlay_labels(axes[3], pred.numpy(), alpha=0.0, linestyle="--")
    axes[3].set_title("contours", fontsize=9)

    mm2 = float(sample["spacing"].prod()) / 100.0  # px -> cm^2
    false_pos, false_neg = draw_error_map(axes[4], pred, label)
    axes[4].set_title(f"errors  +{false_pos * mm2:.2f} / −{false_neg * mm2:.2f} cm²", fontsize=9)

    fig.suptitle(
        f"{sample['key']}  ·  quality={sample['image_quality']}  ·  EF={sample['ef']:.0f}%"
        f"  ·  mean Dice {np.nanmean(list(scores.values())):.3f}"
        f"  ·  {dice_caption(scores)}",
        fontsize=10,
    )
    fig.legend(handles=class_legend() + error_legend(), loc="outside lower center",
               ncol=7, fontsize=8, frameon=False)
    return fig, axes

## 5. One case

In [ ]:
sample = test_ds[0]
pred, probs = predict(sample)
show_prediction(sample, pred, probs)
plt.show()

## 6. Score the whole split

One pass collects the metrics *and* the per-case areas, so the agreement plot in
section 9 costs no extra forward passes. HD95 and ASSD are on: they are slow
enough to be skipped during training, but this is the run where boundary error
is the point.

In [ ]:
evaluator = SegmentationEvaluator(compute_distances=True)
areas = []  # predicted vs expert cavity area, one row per case

with torch.inference_mode():
    for batch in test_loader:
        images = batch["image"].to(DEVICE)
        logits = model(images)
        evaluator.update(
            logits=logits,
            labels=batch["label"],
            spacing=batch["spacing"],
            meta={key: batch[key]
                  for key in ("key", "patient", "view", "instant", "image_quality")
                  if key in batch},
        )

        preds = logits.argmax(1).cpu()
        # spacing is per sample and already corrected for the resize, so these
        # areas are in real cm^2 rather than pixel counts.
        cm2 = batch["spacing"].prod(dim=1) / 100.0
        for i in range(preds.shape[0]):
            areas.append({
                "key": batch["key"][i],
                "view": batch["view"][i],
                "instant": batch["instant"][i],
                "quality": batch["image_quality"][i],
                "true_cm2": float((batch["label"][i] == 1).sum()) * float(cm2[i]),
                "pred_cm2": float((preds[i] == 1).sum()) * float(cm2[i]),
            })

summary = evaluator.compute()
print(f"{SPLIT} split · {CHECKPOINT}\n")
print(evaluator.report(summary))

## 7. Where the errors live

Pooled averages hide the cases that decide whether a model is usable. Poor-quality
images and the 2CH view (where the endocardium is often shadowed) are the usual
weak spots, and they are invisible in a single number.

In [ ]:
def stratify_table(field):
    groups = evaluator.stratified(field)
    width = max(len(str(v)) for v in groups) + 2
    print(f"{field:<{width}}{'n':>5}{'Dice':>8}{'HD95':>9}{'ASSD':>8}")
    print("-" * (width + 30))
    for value, group in groups.items():
        print(f"{str(value):<{width}}{group['n_cases']:>5}{group['mean_dice']:>8.4f}"
              f"{group['mean_hd95']:>9.3f}{group['mean_assd']:>8.3f}")
    print()
    return groups


for field in ("image_quality", "view", "instant"):
    stratify_table(field)

In [ ]:
def dice_frame():
    """Per-case Dice as {class_name: array}, in evaluator case order."""
    return {name: np.array([case.metrics[name]["dice"] for case in evaluator.results])
            for name in evaluator.class_names}


scores = dice_frame()
keys = [case.key for case in evaluator.results]
qualities = [case.meta["image_quality"] for case in evaluator.results]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")

# Distribution per class: the left tail is the story, not the median.
parts = axes[0].violinplot([values[np.isfinite(values)] for values in scores.values()],
                           showmedians=True, widths=0.8)
for body, cls in zip(parts["bodies"], range(1, NUM_CLASSES)):
    body.set_facecolor(CLASS_COLORS[cls]); body.set_alpha(0.55)
axes[0].set_xticks(range(1, len(scores) + 1), list(scores), fontsize=8)
axes[0].set_ylabel("Dice"); axes[0].set_ylim(0, 1.02)
axes[0].set_title(f"per-case Dice ({len(keys)} cases)", fontsize=10)
axes[0].grid(axis="y", alpha=0.25)

# Same cases split by annotated image quality.
order = ["Good", "Medium", "Poor"]
present = [q for q in order if q in set(qualities)] or sorted(set(qualities))
mean_dice = np.nanmean(np.stack(list(scores.values())), axis=0)
axes[1].boxplot([mean_dice[[i for i, q in enumerate(qualities) if q == value]]
                 for value in present], tick_labels=present, showfliers=False)
for x, value in enumerate(present, start=1):
    subset = mean_dice[[i for i, q in enumerate(qualities) if q == value]]
    axes[1].scatter(np.random.default_rng(0).normal(x, 0.05, len(subset)), subset,
                    s=9, alpha=0.45, color="#4a7fe8")
axes[1].set_ylabel("mean Dice"); axes[1].set_ylim(0, 1.02)
axes[1].set_title("mean Dice by annotated image quality", fontsize=10)
axes[1].grid(axis="y", alpha=0.25)
plt.show()

## 8. Worst and best cases

`worst_cases` ranks by class-averaged Dice. Looking at the tail is the fastest
way to tell a fixable data problem (a view the model never learned) from a
genuinely ambiguous image (a shadowed apex no annotator would agree on either).

In [ ]:
def show_cases(case_keys, ncols=4):
    """Compact grid: image with expert contours solid and predicted dashed."""
    nrows = int(np.ceil(len(case_keys) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.6 * nrows),
                            layout="constrained", squeeze=False)

    for ax, key in zip(axes.flat, case_keys):
        s = sample_by_key(test_ds, key)
        p, _ = predict(s)
        ax.imshow(s["image"][0].numpy(), cmap="gray")
        overlay_labels(ax, s["label"].numpy(), alpha=0.0, linestyle="-")
        overlay_labels(ax, p.numpy(), alpha=0.0, linestyle="--")
        case_scores = per_class_dice(p, s["label"])
        ax.set_title(f"{key}  ·  quality={s['image_quality']}\n"
                     f"mean Dice {np.nanmean(list(case_scores.values())):.3f}\n"
                     f"{dice_caption(case_scores)}", fontsize=7, linespacing=1.5)
    for ax in axes.flat:
        ax.set_xticks([]); ax.set_yticks([])
    for ax in axes.flat[len(case_keys):]:
        ax.axis("off")

    fig.legend(handles=class_legend(), loc="outside lower center", ncol=5,
               fontsize=8, frameon=False)
    return fig


worst = evaluator.worst_cases(n=8, metric="dice")
print("worst by mean Dice:")
for key, value in worst:
    print(f"  {key:<28s} {value:.4f}")

show_cases([key for key, _ in worst])
plt.suptitle("worst cases", fontsize=11)
plt.show()

In [ ]:
# The same view on the cases the model handles cleanly, as a reference for what
# "good" looks like on this data.
ranked = sorted(
    ((case.key, float(np.nanmean([case.metrics[n]["dice"] for n in evaluator.class_names])))
     for case in evaluator.results),
    key=lambda kv: kv[1],
    reverse=True,
)
show_cases([key for key, _ in ranked[:4]], ncols=4)
plt.suptitle("best cases", fontsize=11)
plt.show()

## 9. Confidence and boundary error

Softmax output is not calibrated uncertainty, but it is cheap and it localises:
on a well-behaved case the uncertain band is a thin ring on the boundaries, and
on a failure it spreads into a region. Read it against the error map — a model
that is wrong *and* uncertain there is one a reader could triage, while one that
is wrong and confident is not.

In [ ]:
def show_confidence(key, soft_class=1):
    """Prediction, errors, uncertainty, and the soft probability of one class.

    The entropy map and the raw probability of `soft_class` answer different
    questions: entropy shows where the model hesitates between any classes, the
    probability map shows how sharply the decision boundary for that one class
    is drawn - a wide, gradual band means a border the model is effectively
    guessing at, even where the argmax looks decisive.
    """
    s = sample_by_key(test_ds, key)
    p, probs = predict(s)
    image = s["image"][0].numpy()
    # Normalised predictive entropy: 0 = one class certain, 1 = uniform.
    entropy = -(probs * probs.clamp_min(1e-12).log()).sum(0).numpy() / np.log(NUM_CLASSES)

    fig, axes = plt.subplots(1, 4, figsize=(13.5, 3.8), layout="constrained")
    axes[0].imshow(image, cmap="gray")
    overlay_labels(axes[0], p.numpy())
    axes[0].set_title("prediction", fontsize=9)

    axes[1].imshow(image, cmap="gray")
    draw_error_map(axes[1], p, s["label"])
    axes[1].set_title("errors", fontsize=9)

    im = axes[2].imshow(entropy, cmap="magma", vmin=0, vmax=1)
    axes[2].set_title("normalised entropy", fontsize=9)
    fig.colorbar(im, ax=axes[2], fraction=0.046, shrink=0.85)

    im = axes[3].imshow(probs[soft_class].numpy(), cmap="viridis", vmin=0, vmax=1)
    # White is the expert border for this class; the colour transition is the
    # model's, so the gap between them is the boundary error made visible.
    axes[3].contour((s["label"].numpy() == soft_class).astype(float), levels=[0.5],
                    colors=["white"], linewidths=1.0)
    axes[3].set_title(f"P({LABELS[soft_class]})", fontsize=9)
    fig.colorbar(im, ax=axes[3], fraction=0.046, shrink=0.85)

    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"{key}  ·  {dice_caption(per_class_dice(p, s['label']))}", fontsize=10)
    return fig


show_confidence(worst[0][0])   # hardest case
show_confidence(ranked[0][0])  # easiest case
plt.show()

## 10. Does the area agreement hold?

Dice is a similarity score; the clinic reads areas and volumes. A model can hold
a high Dice while shrinking every cavity by a few percent, and that bias survives
straight into ejection fraction. The scatter should sit on the diagonal, and the
residuals should be centred on zero rather than sloped.

In [ ]:
true_cm2 = np.array([row["true_cm2"] for row in areas])
pred_cm2 = np.array([row["pred_cm2"] for row in areas])
diff = pred_cm2 - true_cm2
mean_area = (pred_cm2 + true_cm2) / 2
bias, sd = diff.mean(), diff.std(ddof=1)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), layout="constrained")

limits = [0, max(true_cm2.max(), pred_cm2.max()) * 1.05]
for instant, marker in (("ED", "o"), ("ES", "^")):
    mask = np.array([row["instant"] == instant for row in areas])
    axes[0].scatter(true_cm2[mask], pred_cm2[mask], s=14, alpha=0.55, marker=marker,
                    label=instant)
axes[0].plot(limits, limits, color="0.4", lw=1, ls="--", label="identity")
axes[0].set_xlim(limits); axes[0].set_ylim(limits)
axes[0].set_xlabel("expert LV cavity area (cm²)")
axes[0].set_ylabel("predicted LV cavity area (cm²)")
axes[0].set_title("cavity area agreement", fontsize=10)
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.25)

# Bland-Altman: bias plus 95% limits of agreement.
axes[1].scatter(mean_area, diff, s=14, alpha=0.55, color="#4a7fe8")
axes[1].axhline(bias, color="#e8564a", lw=1.2, label=f"bias {bias:+.2f} cm²")
for sign in (1, -1):
    axes[1].axhline(bias + sign * 1.96 * sd, color="#e8564a", lw=1, ls="--",
                    label="95% limits" if sign == 1 else None)
axes[1].axhline(0, color="0.4", lw=1, ls=":")
axes[1].set_xlabel("mean of expert and predicted area (cm²)")
axes[1].set_ylabel("predicted − expert (cm²)")
axes[1].set_title("Bland–Altman, LV cavity area", fontsize=10)
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.25)
plt.show()

print(f"bias {bias:+.3f} cm²   sd {sd:.3f} cm²   "
      f"95% limits [{bias - 1.96 * sd:+.2f}, {bias + 1.96 * sd:+.2f}] cm²")
print(f"mean absolute error {np.abs(diff).mean():.3f} cm² "
      f"({np.abs(diff).mean() / true_cm2.mean():.1%} of mean expert area)")

## 11. Save the results

Written next to the checkpoint so a run's metrics stay with the weights that
produced them.

In [ ]:
report_path = RUN_DIR / f"{SPLIT}_metrics.json"
report_path.write_text(json.dumps({
    "checkpoint": str(CHECKPOINT),
    "epoch": checkpoint["epoch"],
    "split": SPLIT,
    "summary": summary,
    "stratified": {field: {str(k): v for k, v in evaluator.stratified(field).items()}
                   for field in ("image_quality", "view", "instant")},
    "worst_cases": evaluator.worst_cases(n=20, metric="dice"),
    "lv_area_cm2": {
        "bias": float(bias),
        "sd": float(sd),
        "mae": float(np.abs(diff).mean()),
    },
    "per_case": [
        {"key": case.key,
         "meta": case.meta,
         "metrics": {name: {metric: case.metrics[name][metric] for metric in METRIC_NAMES}
                     for name in evaluator.class_names}}
        for case in evaluator.results
    ],
}, indent=2, default=str))
print(f"wrote {report_path}")